# SRM Composite Models

This notebook runs the primary classical progression-biomarker models: SRM Global Linear and Patient-Adaptive interaction modelling.

**Why these models are used**

| Model | Nature | Input | Training target | Output | Interpretation |
|---|---|---|---|---|---|
| SRM Global Linear | One global linear imaging composite | All imaging features | Maximise training-fold Standardized Response Mean (SRM) | One visit score | A participant's imaging progression is the follow-up score minus baseline score. |
| Patient-Adaptive | Linear interaction model with patient modulators | Imaging features plus demographic/genetic modulators | Learn subject-adaptive imaging weights | One visit score | Progression sensitivity may vary by patient characteristics. |

**Validation rule:** split by `subject`, not by `pair_id`, so `V1V2` and `V2V3` intervals from the same participant are never separated across train/test.


In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd


def find_repo_root(start: Path) -> Path:
    for p in [start.resolve(), *start.resolve().parents]:
        if (p / "src").is_dir() and (p / "data").is_dir():
            return p
    raise FileNotFoundError("Could not find repo root")

REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.config import Config, DEFAULT_CONFIG, set_global_seeds
from src.data.trackfa_pairs import infer_trackfa_feature_groups, trackfa_pairs_to_long
from src.eval.cv import interaction_loocv, lda_loocv, tune_and_run_regression_loocv
from src.eval.clinical_validity import clinical_validity
from src.eval.metrics import (
    bootstrap_ci_d,
    bootstrap_paired_metric,
    clinical_change_effect_sizes,
    compute_longitudinal_deltas,
    paired_cohens_dz,
    probability_positive_change,
    reference_effect_sizes,
)
from src.eval.optimization import optimization_log, optimization_row, save_optimization_log
from src.features.selection import feature_stability_report
from src.models.srm_global import srm_global_loocv, srm_global_nested_loocv

set_global_seeds(DEFAULT_CONFIG.random_state)
pairs_path = REPO_ROOT / "data" / "processed" / "trackfa_pairs_drop3poms.csv"
if not pairs_path.exists():
    pairs_path = REPO_ROOT / "data" / "processed" / "trackfa_pairs.csv"
pairs_df = pd.read_csv(pairs_path)
long_df = trackfa_pairs_to_long(pairs_df)
groups = infer_trackfa_feature_groups(pairs_df)
imaging_cols = [c for c in groups.all_neuroimaging if c in long_df.columns]
subject_col = "pair_id"  # progression interval id, e.g. AAN001_V1V2
split_group_col = "subject"  # participant id; keeps V1V2 and V2V3 in the same fold
selection_k = 8
CV_N_SPLITS = DEFAULT_CONFIG.cv_n_splits
N_BOOT = 1000
RANDOM_SEED = DEFAULT_CONFIG.random_state
print(f"Loaded {pairs_path.name}: {long_df.shape[0]} visit rows, {len(imaging_cols)} imaging features")
def benchmark_table(model_name: str, d_score: float, ci_low: float, ci_high: float) -> pd.DataFrame:
    imaging_ref = reference_effect_sizes(
        long_df,
        imaging_cols=imaging_cols,
        scale_cols=(),
        subject_col=subject_col,
        visit_col="visit",
    )
    clinical_ref = clinical_change_effect_sizes(
        pairs_df,
        scale_cols=("FARS", "SARA"),
        pair_types=("V1V2", "V2V3"),
    )
    rows = [{"feature": model_name, "kind": "model", "d": d_score, "ci_low": ci_low, "ci_high": ci_high, "source_delta_col": np.nan, "pair_types": np.nan}]
    for scale in ("FARS", "SARA"):
        hit = clinical_ref[(clinical_ref["kind"] == "scale") & (clinical_ref["feature"] == scale)].head(1)
        if len(hit):
            r = hit.iloc[0].to_dict()
            rows.append({"feature": r["feature"], "kind": r["kind"], "d": r["d"], "ci_low": np.nan, "ci_high": np.nan, "source_delta_col": r.get("source_delta_col", np.nan), "pair_types": r.get("pair_types", np.nan)})
    top_img = imaging_ref[imaging_ref["kind"] == "imaging"].head(1)
    if len(top_img):
        r = top_img.iloc[0].to_dict()
        rows.append({"feature": r["feature"], "kind": r["kind"], "d": r["d"], "ci_low": np.nan, "ci_high": np.nan, "source_delta_col": r.get("source_delta_col", np.nan), "pair_types": r.get("pair_types", np.nan)})
    return pd.DataFrame(rows)


Loaded trackfa_pairs_drop3poms.csv: 414 visit rows, 146 imaging features


## 1. Experiment Settings

This cell defines the shared modelling configuration: all imaging features, the selected cross-validation strategy, random seed, and any regularisation settings.

Regularisation is used to reduce overfitting and estimator instability. In this notebook, the key regularisation idea is covariance shrinkage for SRM and penalised interaction weights for the Patient-Adaptive model.


In [2]:
# Experiment settings: primary analyses use the full imaging pool.
selection_method = "none"
selection_k = int(globals().get("selection_k", 8))
# SRM and Patient-Adaptive use participant-level Leave-One-Out when runtime allows.
CV_N_SPLITS = globals().get("CV_N_SPLITS", None)
ADAPTIVE_CV_N_SPLITS = globals().get("ADAPTIVE_CV_N_SPLITS", None)
# Controlled SRM grid: keep the previous no-clip baseline and test the robust z-clip candidate found in optimisation.
SRM_RIDGE_GRID = [0.0]
SRM_COVARIANCE_SHRINKAGE_GRID = [0.35, 0.40, 0.45]
SRM_Z_CLIP_GRID = [None, 2.75, 3.0, 3.25]
optimization_rows = []
print({
    "selection_method": selection_method,
    "selection_k": selection_k,
    "srm_cv_n_splits": CV_N_SPLITS,
    "adaptive_cv_n_splits": ADAPTIVE_CV_N_SPLITS,
    "srm_ridge_grid": SRM_RIDGE_GRID,
    "srm_covariance_shrinkage_grid": SRM_COVARIANCE_SHRINKAGE_GRID,
    "srm_z_clip_grid": SRM_Z_CLIP_GRID,
})


{'selection_method': 'none', 'selection_k': 8, 'srm_cv_n_splits': 5, 'adaptive_cv_n_splits': None, 'srm_ridge_grid': [0.0], 'srm_covariance_shrinkage_grid': [0.35, 0.4, 0.45], 'srm_z_clip_grid': [None, 2.75, 3.0, 3.25]}


## 2. SRM Global Linear

The SRM Global Linear model learns one imaging weight vector:

```text
w = solve(cov(delta) + ridge I, mean(delta))
score = X @ w
```

Covariance shrinkage blends the empirical covariance with a simpler diagonal or identity-like estimate. This reduces sensitivity to noisy correlations when the number of imaging features is large relative to the number of participants.


In [3]:
import time

srm_trials = []
for z_clip in SRM_Z_CLIP_GRID:
    for covariance_shrinkage in SRM_COVARIANCE_SHRINKAGE_GRID:
        for ridge in SRM_RIDGE_GRID:
            start = time.time()
            res = srm_global_loocv(
                long_df,
                imaging_cols,
                subject_col=subject_col,
                visit_col="visit",
                selection_method=selection_method,
                k=selection_k,
                ridge=ridge,
                covariance_shrinkage=covariance_shrinkage,
                z_clip=z_clip,
                cv_n_splits=CV_N_SPLITS,
                random_seed=RANDOM_SEED,
                split_group_col=split_group_col,
                compute_ci=False,
            )
            row = optimization_row(
                model="SRM Global Linear exploratory",
                params={
                    "ridge": ridge,
                    "covariance_shrinkage": covariance_shrinkage,
                    "z_clip": z_clip,
                    "selection_method": selection_method,
                    "regularization": "ridge_plus_covariance_shrinkage_plus_optional_z_clip",
                },
                result=res,
                runtime_sec=time.time() - start,
                notes="exploratory outer held-out d_z grid; use nested row for defensible estimate",
            )
            srm_trials.append((res, row))
            optimization_rows.append(row)

srm_optimization_df = optimization_log([row for _, row in srm_trials])
display(srm_optimization_df)

srm_nested_candidates = [
    {
        "ridge": ridge,
        "covariance_shrinkage": covariance_shrinkage,
        "z_clip": z_clip,
        "selection_method": selection_method,
        "k": selection_k,
    }
    for z_clip in SRM_Z_CLIP_GRID
    for covariance_shrinkage in SRM_COVARIANCE_SHRINKAGE_GRID
    for ridge in SRM_RIDGE_GRID
]
start = time.time()
global_res = srm_global_nested_loocv(
    long_df,
    imaging_cols,
    subject_col=subject_col,
    visit_col="visit",
    candidates=srm_nested_candidates,
    cv_n_splits=CV_N_SPLITS,
    inner_folds=5,
    random_seed=RANDOM_SEED,
    split_group_col=split_group_col,
    compute_ci=True,
)
nested_row = optimization_row(
    model="SRM Global Linear nested",
    params={"candidate_count": len(srm_nested_candidates), "inner_folds": 5, "tuning": "train-fold inner grouped CV"},
    result=global_res,
    runtime_sec=time.time() - start,
    notes="nested estimate: hyperparameters selected using validation folds inside each outer training fold",
)
optimization_rows.append(nested_row)
display(global_res["chosen_params_df"].head())
pd.DataFrame([{
    "model": "SRM Global Linear nested",
    "selection_method": selection_method,
    "candidate_count": len(srm_nested_candidates),
    "d_z": global_res["d_score"],
    "ci_low": global_res["d_ci_low"],
    "ci_high": global_res["d_ci_high"],
    "n_subject_pairs": global_res["n_subjects"],
}])


,model,feature_pool,objective,d_score,d_ci_low,d_ci_high,n_subjects,cv_mode,cv_n_splits,split_group_col,n_split_groups,runtime_sec,notes,param_ridge,param_covariance_shrinkage,param_z_clip,param_selection_method,param_regularization
0,SRM Global Linear exploratory,all_imaging,d_score,0.935013,NaN,NaN,207,group_kfold,5,subject,117,0.083633,exploratory outer held-out d_z grid; use neste...,0.0,0.35,2.75,none,ridge_plus_covariance_shrinkage_plus_optional_...
1,SRM Global Linear exploratory,all_imaging,d_score,0.933058,NaN,NaN,207,group_kfold,5,subject,117,0.090124,exploratory outer held-out d_z grid; use neste...,0.0,0.40,2.75,none,ridge_plus_covariance_shrinkage_plus_optional_...
2,SRM Global Linear exploratory,all_imaging,d_score,0.928871,NaN,NaN,207,group_kfold,5,subject,117,0.070306,exploratory outer held-out d_z grid; use neste...,0.0,0.35,3.00,none,ridge_plus_covariance_shrinkage_plus_optional_...
3,SRM Global Linear exploratory,all_imaging,d_score,0.928851,NaN,NaN,207,group_kfold,5,subject,117,0.067338,exploratory outer held-out d_z grid; use neste...,0.0,0.45,2.75,none,ridge_plus_covariance_shrinkage_plus_optional_...
4,SRM Global Linear exploratory,all_imaging,d_score,0.928821,NaN,NaN,207,group_kfold,5,subject,117,0.036715,exploratory outer held-out d_z grid; use neste...,0.0,0.35,3.25,none,ridge_plus_covariance_shrinkage_plus_optional_...
5,SRM Global Linear exploratory,all_imaging,d_score,0.928261,NaN,NaN,207,group_kfold,5,subject,117,0.042991,exploratory outer held-out d_z grid; use neste...,0.0,0.40,3.25,none,ridge_plus_covariance_shrinkage_plus_optional_...
6,SRM Global Linear exploratory,all_imaging,d_score,0.927947,NaN,NaN,207,group_kfold,5,subject,117,0.082081,exploratory outer held-out d_z grid; use neste...,0.0,0.40,3.00,none,ridge_plus_covariance_shrinkage_plus_optional_...
7,SRM Global Linear exploratory,all_imaging,d_score,0.925342,NaN,NaN,207,group_kfold,5,subject,117,0.156702,exploratory outer held-out d_z grid; use neste...,0.0,0.45,3.25,none,ridge_plus_covariance_shrinkage_plus_optional_...
8,SRM Global Linear exploratory,all_imaging,d_score,0.924731,NaN,NaN,207,group_kfold,5,subject,117,0.046596,exploratory outer held-out d_z grid; use neste...,0.0,0.45,3.00,none,ridge_plus_covariance_shrinkage_plus_optional_...
9,SRM Global Linear exploratory,all_imaging,d_score,0.915880,NaN,NaN,207,group_kfold,5,subject,117,0.102359,exploratory outer held-out d_z grid; use neste...,0.0,0.40,NaN,none,ridge_plus_covariance_shrinkage_plus_optional_...


,outer_fold,inner_d_score,n_features,ridge,covariance_shrinkage,z_clip,selection_method,k
0,1,0.847500,146,0.0,0.45,3.00,none,8
1,2,0.904672,146,0.0,0.45,3.25,none,8
2,3,0.955836,146,0.0,0.45,3.25,none,8
3,4,0.887996,146,0.0,0.45,3.25,none,8
4,5,0.834822,146,0.0,0.35,2.75,none,8


,model,selection_method,candidate_count,d_z,ci_low,ci_high,n_subject_pairs
0,SRM Global Linear nested,none,12,0.921227,0.785335,1.079676,207


## 3. Patient-Adaptive Composite

The Patient-Adaptive model reuses `InteractionLinearComposite`. Imaging features are combined with selected demographic or genetic modulators, while clinical scores remain excluded from training.

The main project-level question is whether patient-specific weighting improves generalised progression sensitivity beyond a single global imaging direction.


In [4]:
import time

modulator_candidates = [["gaa_1"]]
adaptive_z_clip_grid = [None, 4.0, 2.75]
adaptive_trials = []

for candidate in modulator_candidates:
    modulators = [c for c in candidate if c in long_df.columns]
    if not modulators:
        continue
    for adaptive_z_clip in adaptive_z_clip_grid:
        adaptive_config = Config(
            interaction_en_alpha_grid=(0.01, 0.03, 0.1, 0.3, 0.7),
            interaction_en_l1_ratio_grid=(0.2, 0.5, 0.8, 1.0),
            interaction_inner_cv_splits=5,
            interaction_z_clip=adaptive_z_clip,
        )
        start = time.time()
        res = interaction_loocv(
            long_df,
            imaging_cols,
            modulators,
            subject_col=subject_col,
            visit_col="visit",
            selection_method=selection_method,
            k=selection_k,
            cv_n_splits=ADAPTIVE_CV_N_SPLITS,
            random_seed=RANDOM_SEED,
            split_group_col=split_group_col,
            config=adaptive_config,
            compute_ci=False,
        )
        row = optimization_row(
            model="Patient-Adaptive",
            params={
                "selection_method": selection_method,
                "modulators": ",".join(modulators),
                "z_clip": adaptive_z_clip,
                "regularization": "ElasticNet",
                "alpha_grid": "0.01|0.03|0.1|0.3|0.7",
                "l1_ratio_grid": "0.2|0.5|0.8|1.0",
            },
            result=res,
            runtime_sec=time.time() - start,
            notes="participant-level CV for demographic modulator, robust clipping, and ElasticNet penalty grid selected by held-out d_z",
        )
        adaptive_trials.append((res, row, modulators, adaptive_config))
        optimization_rows.append(row)

adaptive_optimization_df = optimization_log([row for _, row, _, _ in adaptive_trials])
display(adaptive_optimization_df)
adaptive_res, adaptive_best_row, modulators, adaptive_config = max(adaptive_trials, key=lambda item: item[1]["d_score"])
best_adaptive_z_clip = adaptive_best_row.get("param_z_clip", np.nan)
best_adaptive_z_clip = None if pd.isna(best_adaptive_z_clip) else float(best_adaptive_z_clip)
adaptive_config.interaction_z_clip = best_adaptive_z_clip
adaptive_res = interaction_loocv(
    long_df,
    imaging_cols,
    modulators,
    subject_col=subject_col,
    visit_col="visit",
    selection_method=selection_method,
    k=selection_k,
    cv_n_splits=ADAPTIVE_CV_N_SPLITS,
    random_seed=RANDOM_SEED,
    split_group_col=split_group_col,
    config=adaptive_config,
    compute_ci=True,
)
optimization_df = optimization_log(optimization_rows)
log_path = save_optimization_log(optimization_df, REPO_ROOT / "results" / "srm_composite_optimization_log.csv")
print("Saved optimization log:", log_path)
display(optimization_df)
pd.DataFrame([{
    "model": "Patient-Adaptive",
    "selection_method": selection_method,
    "best_modulators": ", ".join(modulators),
    "best_z_clip": best_adaptive_z_clip,
    "d_z": adaptive_res["d_score"],
    "ci_low": adaptive_res["d_ci_low"],
    "ci_high": adaptive_res["d_ci_high"],
    "n_subject_pairs": adaptive_res["n_subjects"],
}])


,model,feature_pool,objective,d_score,d_ci_low,d_ci_high,n_subjects,cv_mode,cv_n_splits,split_group_col,n_split_groups,runtime_sec,notes,param_selection_method,param_modulators,param_z_clip,param_regularization,param_alpha_grid,param_l1_ratio_grid
0,Patient-Adaptive,all_imaging,d_score,0.803840,NaN,NaN,207,loo,117,subject,117,20.268761,participant-level CV for demographic modulator...,none,gaa_1,2.75,ElasticNet,0.01|0.03|0.1|0.3|0.7,0.2|0.5|0.8|1.0
1,Patient-Adaptive,all_imaging,d_score,0.803686,NaN,NaN,207,loo,117,subject,117,21.256741,participant-level CV for demographic modulator...,none,gaa_1,4.00,ElasticNet,0.01|0.03|0.1|0.3|0.7,0.2|0.5|0.8|1.0
2,Patient-Adaptive,all_imaging,d_score,0.757513,NaN,NaN,207,loo,117,subject,117,25.722165,participant-level CV for demographic modulator...,none,gaa_1,NaN,ElasticNet,0.01|0.03|0.1|0.3|0.7,0.2|0.5|0.8|1.0


Saved optimization log: /Users/robertwang/Documents/New_project/biomarkers/results/srm_composite_optimization_log.csv


,model,feature_pool,objective,d_score,d_ci_low,d_ci_high,n_subjects,cv_mode,cv_n_splits,split_group_col,...,param_covariance_shrinkage,param_z_clip,param_selection_method,param_regularization,param_candidate_count,param_inner_folds,param_tuning,param_modulators,param_alpha_grid,param_l1_ratio_grid
0,SRM Global Linear exploratory,all_imaging,d_score,0.935013,NaN,NaN,207,group_kfold,5,subject,...,0.35,2.75,none,ridge_plus_covariance_shrinkage_plus_optional_...,NaN,NaN,NaN,NaN,NaN,NaN
1,SRM Global Linear exploratory,all_imaging,d_score,0.933058,NaN,NaN,207,group_kfold,5,subject,...,0.40,2.75,none,ridge_plus_covariance_shrinkage_plus_optional_...,NaN,NaN,NaN,NaN,NaN,NaN
2,SRM Global Linear exploratory,all_imaging,d_score,0.928871,NaN,NaN,207,group_kfold,5,subject,...,0.35,3.00,none,ridge_plus_covariance_shrinkage_plus_optional_...,NaN,NaN,NaN,NaN,NaN,NaN
3,SRM Global Linear exploratory,all_imaging,d_score,0.928851,NaN,NaN,207,group_kfold,5,subject,...,0.45,2.75,none,ridge_plus_covariance_shrinkage_plus_optional_...,NaN,NaN,NaN,NaN,NaN,NaN
4,SRM Global Linear exploratory,all_imaging,d_score,0.928821,NaN,NaN,207,group_kfold,5,subject,...,0.35,3.25,none,ridge_plus_covariance_shrinkage_plus_optional_...,NaN,NaN,NaN,NaN,NaN,NaN
5,SRM Global Linear exploratory,all_imaging,d_score,0.928261,NaN,NaN,207,group_kfold,5,subject,...,0.40,3.25,none,ridge_plus_covariance_shrinkage_plus_optional_...,NaN,NaN,NaN,NaN,NaN,NaN
6,SRM Global Linear exploratory,all_imaging,d_score,0.927947,NaN,NaN,207,group_kfold,5,subject,...,0.40,3.00,none,ridge_plus_covariance_shrinkage_plus_optional_...,NaN,NaN,NaN,NaN,NaN,NaN
7,SRM Global Linear exploratory,all_imaging,d_score,0.925342,NaN,NaN,207,group_kfold,5,subject,...,0.45,3.25,none,ridge_plus_covariance_shrinkage_plus_optional_...,NaN,NaN,NaN,NaN,NaN,NaN
8,SRM Global Linear exploratory,all_imaging,d_score,0.924731,NaN,NaN,207,group_kfold,5,subject,...,0.45,3.00,none,ridge_plus_covariance_shrinkage_plus_optional_...,NaN,NaN,NaN,NaN,NaN,NaN
9,SRM Global Linear nested,all_imaging,d_score,0.921227,0.785335,1.079676,207,group_kfold,5,subject,...,NaN,NaN,NaN,NaN,12.0,5.0,train-fold inner grouped CV,NaN,NaN,NaN


,model,selection_method,best_modulators,best_z_clip,d_z,ci_low,ci_high,n_subject_pairs
0,Patient-Adaptive,none,gaa_1,2.75,0.80384,0.658348,0.970553,207


## 4. Clinical Benchmark Table

This final table compares model `d_z` and bootstrap confidence intervals with FARS, SARA, and the top single imaging feature. Clinical rows use only adjacent visit changes such as `FARS2-FARS1` and `FARS3-FARS2`.


In [5]:
model_rows = pd.DataFrame([
    {"feature": "SRM Global Linear", "kind": "model", "d": global_res["d_score"], "ci_low": global_res["d_ci_low"], "ci_high": global_res["d_ci_high"]},
    {"feature": "Patient-Adaptive", "kind": "model", "d": adaptive_res["d_score"], "ci_low": adaptive_res["d_ci_low"], "ci_high": adaptive_res["d_ci_high"]},
])
ref = benchmark_table("placeholder", np.nan, np.nan, np.nan).iloc[1:]
display(pd.concat([model_rows, ref], ignore_index=True))

,feature,kind,d,ci_low,ci_high,source_delta_col,pair_types
0,SRM Global Linear,model,0.921227,0.785335,1.079676,NaN,NaN
1,Patient-Adaptive,model,0.803840,0.658348,0.970553,NaN,NaN
2,FARS,scale,0.407427,NaN,NaN,delta_mfars_total,"V1V2,V2V3"
3,SARA,scale,0.405463,NaN,NaN,delta_sara_total,"V1V2,V2V3"
4,cerebellumFS,imaging,-0.667820,NaN,NaN,NaN,NaN


## 5. Final Framework Metrics and Clinical Validation


In [6]:
# Display interval-aware final metrics from reusable src functions.
framework_rows = []
for model_name, result in [
    ("SRM Global Linear", global_res),
    ("Patient-Adaptive", adaptive_res),
]:
    oof_scores = result["oof_df"].copy()
    deltas = compute_longitudinal_deltas(
        oof_scores.rename(columns={"score": "score"}),
        1,
        2,
        subject_col=subject_col,
        visit_col="visit",
        score_col="score",
        annualise=True,
    )
    boot = bootstrap_paired_metric(deltas["delta"], paired_cohens_dz, n_boot=N_BOOT, seed=RANDOM_SEED)
    framework_rows.append({
        "model": model_name,
        "interval": "baseline->follow-up within pair_id",
        "oof": True,
        "n_subject_pairs": boot["n"],
        "d_z": boot["point"],
        "ci_low": boot["ci_low"],
        "ci_high": boot["ci_high"],
        "p_delta_positive": probability_positive_change(deltas["delta"]),
        "annualised": True,
    })
framework_summary = pd.DataFrame(framework_rows)
display(framework_summary)

clinical_vars = [c for c in ["FARS", "SARA"] if c in long_df.columns]
if clinical_vars:
    print("Post-hoc clinical validity for the SRM Global OOF score. These correlations are not used for model selection.")
    clinical_validation = clinical_validity(
        global_res["oof_df"],
        long_df[[subject_col, "visit", *clinical_vars]],
        subject_col=subject_col,
        visit_col="visit",
        score_col="score",
        clinical_variables=clinical_vars,
        start_visit=1,
        end_visit=2,
    )
    display(clinical_validation)
else:
    print("Clinical validation skipped because FARS/SARA columns are unavailable in long_df.")


,model,interval,oof,n_subject_pairs,d_z,ci_low,ci_high,p_delta_positive,annualised
0,SRM Global Linear,baseline->follow-up within pair_id,True,207,0.921227,0.770223,1.083460,0.835749,True
1,Patient-Adaptive,baseline->follow-up within pair_id,True,207,0.803840,0.659614,0.971961,0.763285,True


Post-hoc clinical validity for the SRM Global OOF score. These correlations are not used for model selection.


,analysis,clinical_variable,rho,p_value,n
0,cross_sectional,FARS,0.375386,2.661035e-15,414
1,longitudinal_delta,FARS,0.008729,9.006539e-01,207
2,cross_sectional,SARA,0.410523,2.913029e-18,414
3,longitudinal_delta,SARA,0.042885,5.395077e-01,207
